# EmbeddingGemma — Dense Semantic Search with vLLM Factory

This notebook demonstrates end-to-end dense embedding search using
`unsloth/embeddinggemma-300m` via the vLLM Factory plugin.

**What you'll see:**
- Index 1,000 documents on a single GPU with continuous batching
- Run semantic search queries with cosine similarity
- Benchmark throughput — tune `dtype`, batch size, and `enforce_eager` for your hardware

## Setup

In [ ]:
import sys
import os
import time
import torch
import numpy as np
from vllm import LLM

ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, ROOT)
sys.path.insert(0, os.path.join(ROOT, "plugins"))
sys.path.insert(0, os.path.join(ROOT, "models"))

## Load Model

In [ ]:
llm = LLM(
    model="unsloth/embeddinggemma-300m",
    trust_remote_code=True,
    enforce_eager=False,  # enable CUDA graphs
    dtype="bfloat16",
    gpu_memory_utilization=0.5,
    disable_log_stats=True,
)
print("✅ Model loaded")

## Create a Document Corpus

In [ ]:
documents = [
    "Machine learning is a branch of artificial intelligence focused on building systems that learn from data.",
    "Deep neural networks use multiple layers of interconnected nodes to learn hierarchical representations.",
    "Natural language processing enables computers to understand, interpret, and generate human language.",
    "Reinforcement learning trains agents to make decisions by rewarding desired behaviors.",
    "Computer vision allows machines to interpret and understand visual information from the world.",
    "Transfer learning leverages knowledge from pre-trained models to improve performance on new tasks.",
    "Generative adversarial networks consist of two neural networks competing against each other.",
    "Attention mechanisms allow models to focus on relevant parts of the input sequence.",
    "Transformer architecture revolutionized NLP by enabling parallel processing of sequences.",
    "Convolutional neural networks are particularly effective for image recognition tasks.",
    "Recurrent neural networks process sequential data by maintaining internal memory states.",
    "Gradient descent optimization iteratively adjusts model parameters to minimize the loss function.",
    "Batch normalization stabilizes training by normalizing layer inputs across mini-batches.",
    "Dropout regularization prevents overfitting by randomly deactivating neurons during training.",
    "Data augmentation increases training set diversity through transformations of existing samples.",
    "Federated learning enables model training across decentralized devices without sharing raw data.",
    "AutoML automates the process of selecting and tuning machine learning algorithms.",
    "Neural architecture search uses AI to design optimal neural network structures.",
    "Knowledge distillation transfers learned representations from large models to smaller ones.",
    "Few-shot learning enables models to generalize from very limited training examples.",
    "The European Central Bank manages monetary policy for the eurozone member states.",
    "Quantum computing uses quantum bits to perform calculations exponentially faster than classical computers.",
    "Climate change is driven primarily by greenhouse gas emissions from human activities.",
    "The human genome contains approximately 3 billion base pairs of DNA.",
    "Blockchain technology provides a decentralized, immutable ledger for recording transactions.",
]

# Scale up to 1000 documents by cycling through with variations
corpus = []
topics = ["AI", "science", "technology", "medicine", "finance", "engineering"]
for i in range(1000):
    base = documents[i % len(documents)]
    corpus.append(f"[Doc {i}] {base}")

print(f"📚 Corpus size: {len(corpus)} documents")

## Index All Documents

In [ ]:
print("Indexing 1,000 documents...")
t0 = time.perf_counter()
outputs = llm.embed(corpus)
elapsed = time.perf_counter() - t0

embeddings = np.array([o.outputs.embedding for o in outputs])
print(f"✅ Indexed {len(corpus)} docs in {elapsed:.2f}s ({len(corpus)/elapsed:.0f} docs/s)")
print(f"   Embedding shape: {embeddings.shape}")

## Semantic Search

In [ ]:
def search(query: str, top_k: int = 5):
    """Embed query and find most similar documents."""
    q_out = llm.embed([query])
    q_vec = np.array(q_out[0].outputs.embedding)

    # Cosine similarity (embeddings are already normalized by the model)
    scores = embeddings @ q_vec
    top_idx = np.argsort(scores)[::-1][:top_k]

    print(f"\n🔍 Query: '{query}'")
    print(f"   Top {top_k} results:")
    for rank, idx in enumerate(top_idx, 1):
        print(f"   {rank}. [{scores[idx]:.4f}] {corpus[idx][:100]}...")


search("How do neural networks learn?")
search("What is blockchain?")
search("Tell me about climate and environment")
search("How does reinforcement learning work?")

## Throughput Benchmark

In [ ]:
batch_sizes = [1, 8, 32]
texts = ["The quick brown fox jumps over the lazy dog. " * 20] * 32  # ~500 tokens each

for bs in batch_sizes:
    batch = texts[:bs]
    # Warmup
    for _ in range(5):
        llm.embed(batch)
    # Timed
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(20):
        llm.embed(batch)
    torch.cuda.synchronize()
    elapsed = (time.perf_counter() - t0) / 20
    print(f"Batch {bs:3d}: {bs/elapsed:.1f} req/s  ({elapsed*1000:.1f}ms)")

## Summary

EmbeddingGemma via vLLM Factory provides:
- Continuous batching and CUDA graph capture for production-grade throughput
- Seamless vLLM integration with `llm.embed()` API
- Tune `dtype`, batch size, and `enforce_eager` for your hardware